In [ ]:
import pandas as pd
from pathlib import Path
import itertools
# import os
# import sys
# sys.path.append(os.path.abspath('..'))
from neutrophil_shape.config.loader import load_config
from neutrophil_shape.CustomFunctions import DetailedBalance, utils

In [ ]:
### load config stuff
config = load_config(microscope_type = 'confocal')
#set alignment
config._alignment = 'trajectory'
savedir = config.common.savedir
datadir = savedir / 'shape_data'
dbdir = savedir / 'detailed_balance'
npcs = config.common.npcs
ntrans = config.db_params.ntrans
origins = config.db_params.origins
pc_combos = itertools.combinations(range(1,npcs+1), 2)
####### load common directories and data
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)

## restrict the treatments and PCs specifically for bootstrapping
bstreats = ['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
bspcs = [[1,2],[4,5],[2,8]]
alldatabs = False

In [ ]:
############# create all CGPSs #############
for whichpcs in pc_combos:

    if __name__ ==  '__main__':
        ########### get raw transitions and pairs ###########
        rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                FullFrame, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ########### interpolate all transitions so that only individual transitions are made ###########
        transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                rawtrans, #pandas dataframe with all of the cgps binned data
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

        ############## get the counts of cells leaving 
        trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                )

Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2561.8040325216034 minutes
Total time observed in this CGPS was 6323.616507275451 minutes
Total time observed in this CGPS was 1438.3734253163423 minutes
Total time observed in this CGPS was 3152.405812876531 minutes
Total time observed in this CGPS was 35.741651884947714 minutes
Total time observed in this CGPS was 1371.7429756430913 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2588.7023176773855 minutes
Total time observed in this CGPS was 6384.537937133702 minutes
Total time observed in this CGPS was 1441.2536577570115 minutes
Total time observed in this CGPS was 3193.123257797886 minutes
Total time observed in this CGPS was 35.248663978390724 minutes
Total time observed in this CGPS was 1369.1586076144174 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating traj

Total time observed in this CGPS was 6496.084408244668 minutes
Total time observed in this CGPS was 1450.4346736754453 minutes
Total time observed in this CGPS was 3246.598966372599 minutes
Total time observed in this CGPS was 35.78881988318933 minutes
Total time observed in this CGPS was 1382.8753345870762 minutes
Finished finding transition rates
Already made this CGPS
Already made this CGPS
Already made this CGPS
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2569.9717904018516 minutes
Total time observed in this CGPS was 6367.210940957724 minutes
Total time observed in this CGPS was 1442.6668404443228 minutes
Total time observed in this CGPS was 3182.064865432874 minutes
Total time observed in this CGPS was 35.2490713680128 minutes
Total time observed in this CGPS was 1369.2977457512404 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 2588.692436

In [ ]:

for whichpcs in pc_combos:
    ############# measure aer and cycling frequencies ###########
    #add specific scaling
    xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
    #set the origin to the actual center
    origin = origins[pc_combos.index(whichpcs)]

    ############### measure aer and cycling frequency for the raw transitions
    #get the area scaling in x and y based on the size of the bins in the cgps
    results = []
    for i, cell in rawtrans.groupby('CellID'):
        #sort data and get continuous transitions in order
        cell, runs = utils.get_consecutive_transitions(cell)
        for r in runs:
            c = cell.iloc[r].reset_index(drop=True)
            results.append(DetailedBalance.get_area_enclosing_rate((
                c,
                config.db_params.nbins,
                xyscaling,
                origin,
                )))

    #make a dataframe and save it
    allaers = pd.concat(results, ignore_index = True)
    allaers.to_csv(dbdir.joinpath(
                    f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))



In [3]:
########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############
for whichpcs in pc_combos:
    #pass if we want to restrict PCs
    if len(bspcs)>0 and whichpcs not in bspcs:
        continue
    ## use a separate savedir to bootstrap using all data
    bssavestr = 'alldatabs' if alldatabs else 'separatedatabs'

    if __name__ ==  '__main__':
        #### open the transitions
        rawtrans = pd.read_csv(dbdir.joinpath(
            f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

        #merge all treatments if bootstrapping with all data
        if alldatabs:
            rawtrans.loc[:,'Treatment'] = 'alldata'

        #restrict to bootstrapped treatments if desired
        if len(bstreats)>0:
            rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]


        ############## BOOTSTRAP MANY TRAJECTORIES ##########
        bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,
                bssavestr, #where to save the bootstrapped dataframes
                )


        ############# open average bootstrapped currents ###################
        bsfield_sep = DetailedBalance.get_avg_current_error(
                bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                config,    
                bssavestr, #where to save the aggregated counts
                )


        ############# measure aer and cycling frequencies ###########
        #add specific scaling
        xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
        #set the origin to the actual center
        center = origins[int(a-1)][int(b-(2+a-1))]

        DetailedBalance.get_aer_cf(
            bstrans,
            xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
            center, #origin in [x bin,y bin]
            whichpcs, #which two PCs to use in the cgps [x,y]
            config,
            bssavestr, #where to save calculated aers and cfs
            )



Boostrapping trajectories with 1 transition samples for CK666


100%|██████████| 3000/3000 [04:46<00:00, 10.48it/s]


Interpolating trajectories for CK666


100%|██████████| 3000/3000 [00:08<00:00, 373.55it/s] 


Calculating bootstrapped CGPS transition rates for CK666


...

100%|██████████| 3000/3000 [00:07<00:00, 376.16it/s] 


Calculating bootstrapped CGPS transition rates for Random


100%|██████████| 3000/3000 [01:36<00:00, 31.25it/s]


Finished bootstrapping


15000it [00:24, 607.71it/s]                         
15000it [01:27, 172.15it/s]                         


In [ ]:
### load config stuff
config = load_config(microscope_type = 'confocal')

for ali in ['trajectory','shape','trajectory_shape']:

    #set alignment
    config._alignment = ali
    savedir = config.common.savedir
    datadir = savedir / 'shape_data'
    dbdir = savedir / 'detailed_balance'
    npcs = config.common.npcs
    ntrans = config.db_params.ntrans
    origins = config.db_params.origins
    ####### load common directories and data
    FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
    centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)

    ## restrict the treatments and PCs specifically for bootstrapping
    bstreats = []#['Random','Galvanotaxis', 'DMSO', 'CK666','Para-Nitro-Blebbistatin']
    bspcs = []#[[1,2]]
    alldatabs = True
    
    
    ############# create all CGPSs #############
    for a in range(1,npcs+1):
        for b in range(1,npcs+1):
            if a == b:
                continue
            elif dbdir.joinpath(f'PC{b}-PC{a}_interpolated_transitions_separated.csv').exists():
                print('Already made this CGPS')
                continue
            else:
                #set the PCs
                whichpcs = [a,b]
                if __name__ ==  '__main__':


                    ########### get raw transitions and pairs ###########
                    rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                            FullFrame, #pandas dataframe with all of the cgps binned data
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,
                            )

                    ########### interpolate all transitions so that only individual transitions are made ###########
                    transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                            rawtrans, #pandas dataframe with all of the cgps binned data
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,
                            )

                    ############## get the counts of cells leaving 
                    trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                            transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,
                            )



    ########### bootstrap transitions and calculate all the aers and cfs around all the pairwise cgps ###############
    for a in range(1,npcs+1):
        for b in range(1,npcs+1):
            #set the PCs
            whichpcs = [a,b]
            #pass if we want to restrict PCs
            if len(bspcs)>0 and whichpcs not in bspcs:
                continue
            ## use a separate savedir to bootstrap using all data
            if alldatabs:
                bssavestr = 'alldatabs'
            else:
                bssavestr = 'separatedatabs'

            if a == b:
                continue
            elif dbdir.joinpath(bssavestr, f'PC{b}-PC{a}_bootstrapped_{ntrans}_transitions.csv').exists():
                print('Already made this plot')
                continue
            else:
                if __name__ ==  '__main__':
                    #### open the transitions
                    rawtrans = pd.read_csv(dbdir.joinpath(
                        f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col=0)

                    #merge all treatments if bootstrapping with all data
                    if alldatabs:
                        rawtrans.loc[:,'Treatment'] = 'alldata'

                    #restrict to bootstrapped treatments if desired
                    if len(bstreats)>0:
                        rawtrans = rawtrans[rawtrans.Treatment.isin(bstreats)]


                    ############## BOOTSTRAP MANY TRAJECTORIES ##########
                    bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                            rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,
                            bssavestr, #where to save the bootstrapped dataframes
                            )


                    ############# open average bootstrapped currents ###################
                    bsfield_sep = DetailedBalance.get_avg_current_error(
                            bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                            whichpcs, #which two PCs to use in the cgps [x,y]
                            config,    
                            bssavestr, #where to save the aggregated counts
                            )


                    ############# measure aer and cycling frequencies ###########
                    #add specific scaling
                    xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
                    #set the origin to the actual center
                    center = origins[int(a-1)][int(b-(2+a-1))]

                    DetailedBalance.get_aer_cf(
                        bstrans,
                        xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
                        center, #origin in [x bin,y bin]
                        whichpcs, #which two PCs to use in the cgps [x,y]
                        config,
                        bssavestr, #where to save calculated aers and cfs
                        )


                    ############### measure aer and cycling frequency for the raw transitions
                    #get the area scaling in x and y based on the size of the bins in the cgps
                    results = []
                    for i, cell in rawtrans.groupby('CellID'):
                        #sort data and get continuous transitions in order
                        cell, runs = utils.get_consecutive_transitions(cell)
                        for r in runs:
                            c = cell.iloc[r].reset_index(drop=True)
                            results.append(DetailedBalance.get_area_enclosing_rate((
                                c,
                                config.db_params.nbins,
                                xyscaling,
                                center,
                                )))

                    #make a dataframe and save it
                    allaers = pd.concat(results, ignore_index = True)
                    allaers.to_csv(dbdir.joinpath(
                                    f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))



KeyboardInterrupt: 

In [3]:
config.db_params.origins

[[[8, 8], [8, 8], [9, 8], [8, 8], [9, 7], [9, 8], [9, 8]],
 [[8, 8], [8, 8], [8, 8], [8, 8], [8, 8], [8, 8]],
 [[8, 8], [8, 8], [8, 8], [8, 8], [8, 8]],
 [[8, 8], [8, 8], [8, 8], [8, 8]],
 [[8, 8], [8, 8], [8, 8]],
 [[6, 8], [8, 8]],
 [[8, 8]]]